# Bayes' Rule

**Audience:** students who have seen conditional probability and the product rule.

**What you will build:** a computational tour of Bayesian updating: prior,
likelihood, evidence and posterior; a medical-testing example; naive Bayes; and
Gaussian conjugacy with the MLE/MAP connection.

## Learning objectives

1. apply $\Pr(Y\mid X)=\Pr(X\mid Y)\Pr(Y)/\Pr(X)$ by hand and in code;
2. explain the normalising constant (evidence) and compute it by marginalisation;
3. use log odds to combine independent pieces of evidence;
4. update a Gaussian prior with Gaussian data;
5. describe MAP as MLE plus a prior penalty.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

rng = np.random.default_rng(0)
plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Setup complete")

## 1. The rule and its vocabulary

$$\underbrace{\Pr(Y=y\mid X=x)}_{\text{posterior}}
= \frac{\underbrace{\Pr(X=x\mid Y=y)}_{\text{likelihood}}
\,\underbrace{\Pr(Y=y)}_{\text{prior}}}{\underbrace{\Pr(X=x)}_{\text{evidence}}}.$$

The evidence does not depend on the hypothesis $y$, so it is just the constant
that makes the posteriors sum to one: $\Pr(X=x)=\sum_{y'}\Pr(X=x\mid Y=y')\Pr(Y=y')$.

In [ ]:
prior = 0.01                 # disease prevalence
sensitivity = 0.99           # P(+ | sick)
false_positive = 0.05        # P(+ | healthy)

p_positive = sensitivity * prior + false_positive * (1 - prior)
posterior = sensitivity * prior / p_positive
print(f"P(disease)            = {prior:.4f}")
print(f"P(+)                  = {p_positive:.4f}")
print(f"P(disease | +)        = {posterior:.4f}")

fig, ax = plt.subplots()
ax.bar(["prior", "posterior"], [prior, posterior], color=["0.6", "tab:red"])
ax.set(ylabel="probability", title="A positive test raises the risk, but not to certainty")
plt.show()

## 2. Evidence is a marginal likelihood

With several hypotheses, multiply prior by likelihood for each one, sum the
results to get the evidence, then divide. This is exactly the same computation as
the medical test, written in vector form.

In [ ]:
hypotheses = np.array([0.2, 0.5, 0.3])
likelihoods = np.array([0.1, 0.8, 0.4])
unnormalised = hypotheses * likelihoods
evidence = unnormalised.sum()
posteriors = unnormalised / evidence

print(f"evidence = {evidence:.4f}, posteriors sum to {posteriors.sum():.4f}")
x = np.arange(len(hypotheses))
fig, ax = plt.subplots()
ax.bar(x - 0.2, hypotheses, width=0.4, label="prior")
ax.bar(x + 0.2, posteriors, width=0.4, label="posterior")
ax.set(xticks=x, xticklabels=[f"H{i}" for i in range(len(hypotheses))],
       ylabel="probability", title="Bayesian updating across hypotheses")
ax.legend()
plt.show()

## 3. Naive Bayes and log odds

Under conditional independence of features given the class, the posterior odds
are the prior odds times a product of likelihood ratios. Working in log odds
turns the product into a sum and avoids underflow:

$$\log\frac{\Pr(S\mid w_1,w_2)}{1-\Pr(S\mid w_1,w_2)}
= \log\frac{\Pr(S)}{1-\Pr(S)} + \sum_j \log\frac{\Pr(w_j\mid S)}{\Pr(w_j\mid\neg S)}.$$

In [ ]:
prior_spam = 0.10
p_free_spam, p_free_ham = 0.50, 0.05
p_money_spam, p_money_ham = 0.40, 0.02

log_odds = (
    np.log(prior_spam / (1 - prior_spam))
    + np.log(p_free_spam / p_free_ham)
    + np.log(p_money_spam / p_money_ham)
)
posterior_spam = 1 / (1 + np.exp(-log_odds))
print(f"prior odds (log) = {np.log(prior_spam/(1-prior_spam)):.3f}")
print(f"posterior odds (log) = {log_odds:.3f}")
print(f"P(spam | 'free', 'money') = {posterior_spam:.4f}")

## 4. Gaussian prior, Gaussian likelihood

If $\mu\sim\mathcal{N}(\mu_0,\tau^2)$ and $x\mid\mu\sim\mathcal{N}(\mu,\sigma^2)$,
then the posterior is Gaussian with

$$\tau_n^2 = \left(\tau^{-2} + \sigma^{-2}\right)^{-1}, \qquad
\mu_n = \tau_n^2\left(\mu_0/\tau^2 + x/\sigma^2\right).$$

The posterior precision is the sum of prior and data precisions: a weighted
average of the prior mean and the observation.

In [ ]:
def gaussian_posterior(prior_mu, prior_var, obs, obs_var):
    post_var = 1 / (1 / prior_var + 1 / obs_var)
    post_mu = post_var * (prior_mu / prior_var + obs / obs_var)
    return post_mu, post_var


prior_mu, prior_var = 0.0, 2.0
obs, obs_var = 2.0, 1.0
post_mu, post_var = gaussian_posterior(prior_mu, prior_var, obs, obs_var)

grid = np.linspace(-5, 7, 500)
fig, ax = plt.subplots()
ax.plot(grid, norm.pdf(grid, prior_mu, np.sqrt(prior_var)), label="prior")
ax.plot(grid, norm.pdf(grid, obs, np.sqrt(obs_var)), label="likelihood")
ax.plot(grid, norm.pdf(grid, post_mu, np.sqrt(post_var)), label="posterior", lw=2)
ax.set(xlabel="mu", ylabel="density", title="Bayesian update for a Gaussian mean")
ax.legend()
plt.show()
print(f"posterior mean = {post_mu:.3f}, variance = {post_var:.3f}")

## 5. MAP interpolation between prior and data

As the prior variance grows, its precision $1/\tau^2$ vanishes and the posterior
mean approaches the observation $\bar x$: MAP becomes MLE. A tight prior pulls
the estimate towards $\mu_0$. In general the MAP estimate maximises
$\log p(\text{data}\mid\theta)+\log p(\theta)$, i.e. MLE with a penalty; an
$\ell_2$ penalty is a Gaussian prior and $\ell_1$ a Laplace prior.

In [ ]:
prior_vars = np.array([0.1, 0.5, 1.0, 5.0, 50.0, 1000.0])
means = np.array([gaussian_posterior(0.0, v, obs, obs_var)[0] for v in prior_vars])

for v, m in zip(prior_vars, means):
    print(f"prior variance {v:8.1f} -> posterior mean {m:.4f}")

fig, ax = plt.subplots()
ax.semilogx(prior_vars, means, marker="o")
ax.axhline(obs, color="tab:red", ls="--", label="MLE (observation)")
ax.set(xlabel="prior variance", ylabel="posterior mean",
       title="MAP approaches MLE as the prior weakens")
ax.legend()
plt.show()

## Exercises

1. Recompute the medical-test posterior with sensitivity $0.95$ and prevalence
   $0.05$. Why does it change so much?
2. Verify that the posteriors in the three-hypothesis example sum to one.
3. Add a third word to the spam model and recompute the log odds.
4. Add a second Gaussian observation and check that the posterior variance drops.
5. Derive the MAP estimate for a Laplace prior on a scalar mean.